In [40]:
#remove when converting to .py file
from pathlib import Path
import importlib.util

PROJECT_ROOT = Path.cwd()
helper_path = PROJECT_ROOT / ".." /"src" / "utils.py"
spec = importlib.util.spec_from_file_location("utils", helper_path)
utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(utils)

In [ ]:
import numpy as np
import librosa
import torch
from transformers import pipeline, ClapProcessor, ClapModel
import json
print(np.isnan(0.0))


False


In [42]:
torch.backends.cudnn.enabled = True
if torch.cuda.is_available():
    print("GPU(s):", torch.cuda.device_count(), torch.cuda.get_device_name(0))  #you need cuda otherwise set device to cpu

GPU(s): 1 NVIDIA GeForce RTX 4060 Laptop GPU


In [43]:
audio_classifier = pipeline(task="zero-shot-audio-classification", model="laion/larger_clap_general", batch=8, device='cuda')   #SET IT HERE

Device set to use cuda


In [47]:
model = ClapModel.from_pretrained("laion/larger_clap_general")
processor = ClapProcessor.from_pretrained("laion/larger_clap_general")

In [44]:
audio_segments_path = '../segments/testSong'
audio = "../audio/testSong.mp3"

In [45]:
clap_label_json = "../json/clap_labels.json"
with open(clap_label_json, 'r') as f:
    music_labels = json.load(f)

anchor_labels_json = "../json/anchor_labels.json"
with open(anchor_labels_json, 'r') as f:
    anchor_labels = json.load(f)

time_segments_json = "../json/k_beat_segments.json"
with open(time_segments_json, 'r') as f:
    k_beats_segments = json.load(f)

In [46]:
result = []
threshold = 0.1 
k = 3   #select top 3 labels
y, sr = librosa.load(audio, sr=22050)
for [start, end] in k_beats_segments:
    chunk = y[int(round(start * sr)): int(round(end * sr))]     #select chunk
    features = {}
    for label in music_labels:
        classes = music_labels[label]
        predictions = audio_classifier(y, candidate_labels = classes)

        top_preds = sorted(predictions, key=lambda x: x['score'], reverse=True)
        filtered = top_preds[:k]
        features[label] = filtered

    result.append({
        "start": start,
        "end": end,
        "feature": features
    })

utils.save_as_json("clap_results", result)

In [48]:
inputs = processor(text=anchor_labels, return_tensors="pt", padding=True, truncation=True)
emb = model.get_text_features(**inputs)
emb = torch.nn.functional.normalize(emb, dim=-1)
with torch.no_grad():
    text_embeds = model.get_text_features(**inputs)
    text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)


In [49]:
output_list = [
    {
        "mood": label,
        "embed": text_embeds[i].cpu().tolist()
    }
    for i, label in enumerate(anchor_labels)
]
utils.save_as_json("anchor_labels_embed", output_list)


In [ ]:
#